# Audit Kualitas Data pada Prediksi TMA SSDS 2026

Notebook ini membangun sebuah penilai kualitas data tanpa pengawasan untuk periode test.
Tujuannya menemukan baris yang perilakunya paling tidak wajar, sebagai kandidat kesalahan
pencatatan yang layak diperiksa lebih lanjut.

Semua langkah dibangun ulang dari nol hanya menggunakan data resmi kompetisi di folder
`data/` dan `data_pendukung/`. Tidak ada nilai jawaban test yang dipakai. Seluruh fungsi
didefinisikan langsung di dalam notebook ini agar mandiri dan mudah ditelusuri.

Ide intinya sederhana. Kita latih sebuah model ringan untuk memperkirakan tinggi muka air
di tiap pos, lalu kita ukur seberapa jauh tiap baris menyimpang dari perilaku wajarnya.
Baris yang menyimpang paling ekstrem, muncul sesaat, dan berada di pos yang secara historis
rawan salah catat akan mendapat skor tertinggi dan naik ke puncak daftar untuk ditinjau.

## 1. Pengaturan dan pustaka

Kita muat pustaka standar analisis data dan model, lalu tetapkan seluruh parameter yang
membuat hasil dapat direproduksi persis. Seed dan hyperparameter dikunci di sini.

In [1]:
import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.signal import lfilter
import geopandas as gpd
from shapely.geometry import Point

# Lokasi data resmi. Ubah bila struktur folder berbeda.
ROOT = "."
DATA = os.path.join(ROOT, "data")
SUP = os.path.join(ROOT, "data_pendukung")

# Batas akhir data latih dan jendela periode test.
CUTOFF = pd.Timestamp("2025-09-18 18:00")

# Parameter model dan skor outlier. Dikunci agar hasil deterministik.
SEED = 3
LGB_PARAMS = dict(num_leaves=3, n_estimators=100, learning_rate=0.05, feature_fraction=0.6)
ANCHOR_BW = 60.0        # lebar penghalusan klimatologi hari dalam tahun
TRANSIENT_K = 1         # lebar jendela tetangga waktu
TRANSIENT_ALPHA = 0.5   # bobot koreksi transien
RISK_FLOOR = 0.02       # lantai risiko agar pos tanpa riwayat tidak nol mutlak
RISK_BETA = 2.0         # penguat pengaruh risiko historis

SLOTS = (6, 12, 18)     # tiga waktu baca harian
print("Siap. Batas data latih:", CUTOFF)

Siap. Batas data latih: 2025-09-18 18:00:00


## 2. Peta sungai dari shapefile resmi

Fitur sungai dibangun dari shapefile HydroRIVERS di `data_pendukung/`. Kita tempelkan tiap
pos ke ruas sungai terdekat, lalu turunkan tiga hal: profil tiap pos (ordo sungai, luas
daerah tangkapan, jarak ke hilir, sistem sungai induk), daftar pasangan pos hulu ke hilir,
dan matriks jarak antar pos. Ketiganya dihitung hidup di sini, bukan dimuat dari file jadi.

In [2]:
def build_geography():
    # Baca koordinat pos dan shapefile sungai, dibatasi kotak wilayah DAS agar ringan.
    ko = pd.read_csv(os.path.join(SUP, "koordinat_pos.csv"))
    shp = None
    for r, _, fs in os.walk(os.path.join(SUP, "HydroRIVERS_v10_au_shp")):
        for f in fs:
            if f.endswith(".shp"):
                shp = os.path.join(r, f)
    gdf = gpd.read_file(shp, bbox=(110.5, -8.5, 112.9, -6.7))

    # Tempelkan tiap pos ke ruas sungai terdekat pada proyeksi meter.
    pts = gpd.GeoDataFrame(ko, geometry=[Point(xy) for xy in zip(ko["longitude"], ko["latitude"])],
                           crs="EPSG:4326")
    cols = ["HYRIV_ID", "NEXT_DOWN", "MAIN_RIV", "ORD_STRA", "UPLAND_SKM", "DIST_DN_KM", "geometry"]
    snap = gpd.sjoin_nearest(pts.to_crs(32749), gdf.to_crs(32749)[cols], distance_col="snap_m")
    snap = snap.sort_values("snap_m").drop_duplicates("nama_pos")

    # Telusuri rantai hilir tiap pos untuk menemukan pasangan pos hulu ke hilir.
    id2next = dict(zip(gdf["HYRIV_ID"], gdf["NEXT_DOWN"]))
    reach2pos = dict(zip(snap["HYRIV_ID"], snap["nama_pos"]))
    pairs = []
    for _, r in snap.iterrows():
        cur, nxt, seen = r["HYRIV_ID"], id2next.get(r["HYRIV_ID"], 0), 0
        while nxt and nxt in id2next and seen < 2000:
            if nxt in reach2pos and reach2pos[nxt] != r["nama_pos"]:
                pairs.append((r["nama_pos"], reach2pos[nxt]))
            cur, nxt, seen = nxt, id2next.get(nxt, 0), seen + 1

    # Tabel statis per pos. Gabung profil sungai, koordinat, dan dua kolom tetap lingkungan.
    dl0 = pd.read_csv(os.path.join(SUP, "data_lingkungan.csv"),
                      usecols=["nama_pos", "built_surface_m2", "landcover_class"]).drop_duplicates("nama_pos")
    static = (snap[["nama_pos", "UPLAND_SKM", "ORD_STRA", "DIST_DN_KM", "MAIN_RIV"]]
              .merge(ko, on="nama_pos").merge(dl0, on="nama_pos").set_index("nama_pos"))
    static["log_upland"] = np.log10(static["UPLAND_SKM"])

    # Peta pos hilir ke daftar pos hulunya.
    upmap = {}
    for hulu, hilir in pairs:
        upmap.setdefault(hilir, []).append(hulu)

    # Matriks jarak antar pos dalam kilometer memakai rumus haversine.
    names = ko["nama_pos"].tolist()
    la = np.radians(ko.set_index("nama_pos")["latitude"])
    lo = np.radians(ko.set_index("nama_pos")["longitude"])
    D = pd.DataFrame(0.0, index=names, columns=names)
    for i, a in enumerate(names):
        for b in names[i + 1:]:
            km = 2 * 6371 * np.arcsin(np.sqrt(np.sin((la[b] - la[a]) / 2) ** 2 +
                 np.cos(la[a]) * np.cos(la[b]) * np.sin((lo[b] - lo[a]) / 2) ** 2))
            D.loc[a, b] = D.loc[b, a] = km
    D = D.round(2)   # bulatkan dua desimal agar bobot tetangga tetap stabil dan hasil reproducible
    return static, upmap, D

STATIC, UPMAP, DIST = build_geography()
print("Pos ter-snap:", len(STATIC), "| pasangan hulu-hilir:", sum(len(v) for v in UPMAP.values()))

Pos ter-snap: 30 | pasangan hulu-hilir: 150


## 3. Rekayasa fitur lingkungan

Dari data lingkungan per jam kita bangun fitur untuk tiap pos pada tiap waktu baca. Fitur
mencakup hujan pada berbagai jendela waktu, kelembapan tanah empat lapis, cuaca, indeks
iklim, akumulasi hujan gaya resesi, keseimbangan air, agregasi hujan sepanjang daerah hulu,
efek waktu tempuh air dari hulu, hujan dari pos tetangga, dan penanda kalender. Fitur hanya
menggambarkan kondisi lingkungan dan tidak pernah menyentuh nilai tinggi muka air.

In [3]:
def load_hourly():
    # Muat kolom lingkungan yang dipakai, urutkan, lalu isi maju celah kecil per pos.
    use = ["datetime", "nama_pos", "rainfall_mm", "humidity_pct", "cloud_cover_pct",
           "temperature_c", "wind_speed_kmh", "dew_point_c",
           "soil_moisture_0_7cm", "soil_moisture_7_28cm", "soil_moisture_28_100cm",
           "soil_moisture_100_255cm", "surface_pressure_hpa", "rmm1", "rmm2",
           "mjo_amplitude", "nino_34"]
    dl = pd.read_csv(os.path.join(SUP, "data_lingkungan.csv"), parse_dates=["datetime"], usecols=use)
    dl = dl.sort_values(["nama_pos", "datetime"])
    dl[use[2:]] = dl.groupby("nama_pos")[use[2:]].ffill()
    return dl


def slot_grid():
    # Grid tiga waktu baca harian dari awal data sampai akhir periode test.
    days = pd.date_range("2023-01-01", "2026-05-18", freq="D")
    return pd.DatetimeIndex([d + pd.Timedelta(hours=h) for d in days for h in SLOTS]).sort_values()


def build_features(dl):
    grid = slot_grid()
    pos_list = sorted(dl["nama_pos"].unique())
    area = STATIC["UPLAND_SKM"]
    # Matriks hujan dan kelembapan tanah per jam untuk agregasi lintas pos yang cepat.
    rain_h = dl.pivot(index="datetime", columns="nama_pos", values="rainfall_mm").sort_index()
    sm100_h = dl.pivot(index="datetime", columns="nama_pos", values="soil_moisture_28_100cm").sort_index()
    sm7_h = dl.pivot(index="datetime", columns="nama_pos", values="soil_moisture_0_7cm").sort_index()

    # ---- Fitur di titik tiap pos ----
    frames = []
    for pos in pos_list:
        g = dl[dl["nama_pos"] == pos].set_index("datetime")
        f = pd.DataFrame(index=g.index)
        rain = g["rainfall_mm"]; c = rain.cumsum()
        win = lambda h: c - c.shift(h)     # jumlah hujan pada h jam terakhir
        f["rain_6h"], f["rain_24h"] = win(6), win(24)
        f["api_3d"], f["api_7d"], f["api_14d"] = win(72), win(168), win(336)
        f["api_30d"], f["api_60d"], f["api_90d"] = win(720), win(1440), win(2160)
        for col, s in [("soil_moisture_0_7cm", "sm7"), ("soil_moisture_7_28cm", "sm28"),
                       ("soil_moisture_28_100cm", "sm100"), ("soil_moisture_100_255cm", "sm255")]:
            f[s] = g[col]
            f[f"{s}_d7"] = g[col] - g[col].shift(168)
            f[f"{s}_d30"] = g[col] - g[col].shift(720)
        for col, s in [("temperature_c", "temp"), ("humidity_pct", "hum"), ("cloud_cover_pct", "cloud"),
                       ("wind_speed_kmh", "wind"), ("surface_pressure_hpa", "pres")]:
            f[f"{s}_24h"] = g[col].rolling(24).mean()
        f["temp_7d"] = g["temperature_c"].rolling(168).mean()
        f["hum_7d"] = g["humidity_pct"].rolling(168).mean()

        # Akumulasi hujan gaya resesi eksponensial dan keseimbangan air.
        for k, nm in [(0.90, "k90"), (0.95, "k95"), (0.98, "k98"), (0.995, "k995")]:
            f[f"api_rec_{nm}"] = lfilter([1.0], [1.0, -k], rain.fillna(0).values)
        et = np.maximum(0.0023 * (g["temperature_c"] + 17.8) *
                        np.sqrt(np.maximum(g["temperature_c"] - g["dew_point_c"], 0)) * 15, 0) / 24
        f["et_24h"] = et.rolling(24).sum()
        wb = (rain.fillna(0) - et.fillna(0)).values
        f["wbal_k98"] = lfilter([1.0], [1.0, -0.98], wb)
        f["wbal_k995"] = lfilter([1.0], [1.0, -0.995], wb)
        f["rain24_x_sm100"] = f["rain_24h"] * g["soil_moisture_28_100cm"]
        f["rain24_x_sm7"] = f["rain_24h"] * g["soil_moisture_0_7cm"]
        f["api7_x_sm100"] = f["api_7d"] * g["soil_moisture_28_100cm"]
        f["sm255_rank"] = g["soil_moisture_100_255cm"].rolling(24 * 365, min_periods=24 * 30).rank(pct=True)
        f["sm100_rank"] = g["soil_moisture_28_100cm"].rolling(24 * 365, min_periods=24 * 30).rank(pct=True)

        # Transformasi nonlinier dan penanda kekeringan atau hujan lebat.
        f["log_rain24"], f["log_api7"], f["log_api30"] = np.log1p(f["rain_24h"]), np.log1p(f["api_7d"]), np.log1p(f["api_30d"])
        f["rain_max_h_24"], f["rain_max_h_72"] = rain.rolling(24).max(), rain.rolling(72).max()
        f["wet_hours_24"] = (rain > 0.1).rolling(24).sum()
        f["wet_hours_168"] = (rain > 0.1).rolling(168).sum()
        f["heavy_hours_168"] = (rain > 2.0).rolling(168).sum()
        dry = (rain <= 0.1).astype(int)
        f["dry_spell_h"] = dry.groupby((dry == 0).cumsum()).cumsum()

        # Indeks iklim skala besar.
        f["nino"] = g["nino_34"]
        f["nino_lag30d"], f["nino_lag90d"] = g["nino_34"].shift(720), g["nino_34"].shift(2160)
        f["mjo_amp"] = g["mjo_amplitude"]
        ph = np.arctan2(g["rmm2"], g["rmm1"])
        f["mjo_sin"], f["mjo_cos"] = np.sin(ph), np.cos(ph)
        f["nama_pos"] = pos
        frames.append(f[f.index.hour.isin(SLOTS)].reset_index())   # saring ke slot lebih awal, hemat memori
    P = pd.concat(frames, ignore_index=True)

    # ---- Agregasi hujan dan tanah sepanjang daerah hulu ----
    catch = []
    for pos in pos_list:
        ups = [u for u in UPMAP.get(pos, []) + [pos] if u in rain_h.columns]
        w = area.reindex(ups).fillna(area.median()).values; w = w / w.sum()
        r_mean, r_wsum = rain_h[ups].mean(axis=1), (rain_h[ups] * w).sum(axis=1)
        sm_mean, sm7_mean = sm100_h[ups].mean(axis=1), sm7_h[ups].mean(axis=1)
        cm, cw = r_mean.cumsum(), r_wsum.cumsum()
        d = pd.DataFrame(index=rain_h.index)
        for h, nm in [(24, "24h"), (168, "7d"), (336, "14d"), (720, "30d"), (2160, "90d")]:
            d[f"up_api_{nm}"], d[f"upw_api_{nm}"] = cm - cm.shift(h), cw - cw.shift(h)
        d["up_api_rec_k98"] = lfilter([1.0], [1.0, -0.98], r_wsum.fillna(0).values)
        d["up_sm100"], d["up_sm7"] = sm_mean, sm7_mean
        d["up_sm100_d30"] = sm_mean - sm_mean.shift(720)
        d["n_upstream"] = len(ups); d["nama_pos"] = pos
        catch.append(d[d.index.hour.isin(SLOTS)].reset_index())
    C = pd.concat(catch, ignore_index=True)

    # ---- Hujan hulu yang digeser sesuai perkiraan waktu tempuh air ----
    dist_dn = STATIC["DIST_DN_KM"]; route = []
    for pos in pos_list:
        d = pd.DataFrame(index=rain_h.index)
        for vel, tag in [(50, "v50"), (100, "v100"), (200, "v200")]:
            acc, tot = pd.Series(0.0, index=rain_h.index), 0.0
            for u in UPMAP.get(pos, []):
                if u not in rain_h.columns:
                    continue
                dd = float(dist_dn.get(u, np.nan) - dist_dn.get(pos, np.nan))
                if not np.isfinite(dd) or dd <= 0:
                    continue
                lag = int(np.clip(round(dd / vel * 24), 0, 24 * 10))
                wu = float(area.get(u, area.median()))
                acc = acc.add(rain_h[u].shift(lag) * wu, fill_value=0.0); tot += wu
            if tot > 0:
                acc = acc / tot
            cc = acc.cumsum()
            d[f"route_{tag}_24h"], d[f"route_{tag}_7d"], d[f"route_{tag}_14d"] = cc - cc.shift(24), cc - cc.shift(168), cc - cc.shift(336)
        d["nama_pos"] = pos
        route.append(d[d.index.hour.isin(SLOTS)].reset_index())
    R = pd.concat(route, ignore_index=True)

    # ---- Hujan dari lima pos tetangga terdekat berbobot jarak ----
    neigh = []
    for pos in pos_list:
        near = DIST.loc[pos].drop(pos).astype(float).nsmallest(5)
        w = 1.0 / (near.values + 5.0); w = w / w.sum()
        cols = [c for c in near.index if c in rain_h.columns]
        rn = (rain_h[cols] * w[:len(cols)]).sum(axis=1); cc = rn.cumsum()
        d = pd.DataFrame(index=rain_h.index)
        d["neigh_api_24h"], d["neigh_api_7d"], d["neigh_api_30d"] = cc - cc.shift(24), cc - cc.shift(168), cc - cc.shift(720)
        d["nama_pos"] = pos
        neigh.append(d[d.index.hour.isin(SLOTS)].reset_index())
    Nb = pd.concat(neigh, ignore_index=True)

    # ---- Gabung semua, saring ke grid waktu baca, tambah kalender dan fitur statis ----
    F = (P.merge(C, on=["datetime", "nama_pos"], how="left")
           .merge(R, on=["datetime", "nama_pos"], how="left")
           .merge(Nb, on=["datetime", "nama_pos"], how="left"))
    F = F[F["datetime"].isin(grid)].reset_index(drop=True)
    F["month"], F["hour"] = F["datetime"].dt.month, F["datetime"].dt.hour
    doy = F["datetime"].dt.dayofyear
    for k in (1, 2, 3):
        F[f"doy_sin{k}"], F[f"doy_cos{k}"] = np.sin(2 * np.pi * k * doy / 365.25), np.cos(2 * np.pi * k * doy / 365.25)
    F["doy"] = doy
    F = F.merge(STATIC.drop(columns=["UPLAND_SKM"]), left_on="nama_pos", right_index=True, how="left")
    return F

import gc
DL = load_hourly()
BASE = build_features(DL)
del DL; gc.collect()          # bebaskan data per jam yang sudah tidak dipakai
print("Tabel fitur:", BASE.shape)

Tabel fitur: (111060, 100)


## 4. Target bersih, jangkar musiman, dan risiko pos

Untuk melatih model kita butuh nilai target dari data latih. Sebelum dipakai, target
dibersihkan dari salah catat yang jelas keliru memakai aturan sederhana. Kita juga hitung
jangkar musiman, yaitu perkiraan tinggi air normal untuk tiap hari dalam setahun yang
dihaluskan agar mulus. Target model adalah selisih tinggi air terhadap jangkar ini.

Terakhir kita hitung risiko tiap pos, yaitu seberapa sering pos itu tercatat salah input
di masa lalu. Pos yang sering keliru mendapat risiko tinggi. Angka ini murni dari data latih.

In [4]:
def spike_flag(g):
    # Tandai baris yang menyimpang jauh dari median lingkungan sekitarnya sebagai salah catat.
    s = g["tma_mdpl"]
    med = s.rolling(9, center=True, min_periods=3).median()
    mad = (s - med).abs().rolling(9, center=True, min_periods=3).median()
    return (s - med).abs() > np.maximum(10 * mad, 3.0)


def clean_train():
    # Muat data latih, tandai salah catat, dan buang periode awal dua pos khusus.
    tr = pd.read_csv(os.path.join(DATA, "train.csv"), parse_dates=["datetime"]).sort_values(["nama_pos", "datetime"])
    tr["spike"] = tr.groupby("nama_pos", group_keys=False).apply(spike_flag, include_groups=False).values
    drop = tr["spike"].copy()
    drop |= (tr["nama_pos"] == "Kali Anyar - Kreteg Abang") & (tr["datetime"] < "2024-03-01")
    drop |= (tr["nama_pos"] == "Gunungsari") & (tr["datetime"] < "2024-12-01")
    clean = tr[~drop].drop(columns=["spike"]).reset_index(drop=True)
    return clean, tr


def anchor_doy(clean, target_df, bw=ANCHOR_BW):
    # Klimatologi hari dalam tahun yang dihaluskan dengan kernel gaussian melingkar per pos.
    up = clean[clean["datetime"] <= CUTOFF].copy()
    up["doy"] = up["datetime"].dt.dayofyear
    tgt_doy = target_df["datetime"].dt.dayofyear.values
    pos_arr = target_df["nama_pos"].values
    out = np.full(len(target_df), np.nan)
    for pos, g in up.groupby("nama_pos"):
        m = pos_arr == pos
        if not m.any():
            continue
        d, v = g["doy"].values.astype(float), g["tma_mdpl"].values
        diff = np.abs(tgt_doy[m].astype(float)[:, None] - d[None, :])
        diff = np.minimum(diff, 365.25 - diff)
        w = np.exp(-0.5 * (diff / bw) ** 2)
        out[m] = (w @ v) / np.maximum(w.sum(1), 1e-9)
    return out


def per_pos_risk(raw_train):
    # Frekuensi historis salah catat tiap pos, dihitung dari data latih.
    raw = raw_train.sort_values(["nama_pos", "datetime"]).copy()
    raw["spike"] = raw.groupby("nama_pos", group_keys=False).apply(spike_flag, include_groups=False).values
    return raw.groupby("nama_pos")["spike"].mean()

CLEAN, RAW = clean_train()
RISK = per_pos_risk(RAW)
print("Pos dengan risiko salah catat tertinggi:", RISK.idxmax(), round(float(RISK.max()), 4))

Pos dengan risiko salah catat tertinggi: Kali Pepe - Tugu Boto 0.0028


## 5. Rakit data latih dan baris test

Data latih adalah gabungan target bersih dengan fitur lingkungan. Baris test diambil dari
daftar id resmi di `data/test.csv`, lalu ditempeli fitur yang sama. Perhatikan bahwa kita
tidak pernah memuat nilai jawaban test. Kolom yang kita butuhkan dari baris test hanyalah
identitas pos dan waktu, agar model bisa memberi perkiraan lalu kita ukur simpangannya.

Urutan daftar fitur dikunci agar hasil model persis sama setiap dijalankan.

In [5]:
# Urutan kelompok fitur dikunci. Ini menentukan urutan kolom yang masuk ke model.
FEAT_GROUPS = {
    "point": ["rain_6h", "rain_24h", "api_3d", "api_7d", "api_14d", "api_30d", "api_60d", "api_90d",
              "sm7", "sm7_d7", "sm7_d30", "sm28", "sm28_d7", "sm28_d30", "sm100", "sm100_d7",
              "sm100_d30", "sm255", "sm255_d7", "sm255_d30", "temp_24h", "hum_24h", "cloud_24h",
              "wind_24h", "pres_24h", "temp_7d", "hum_7d"],
    "hydro": ["api_rec_k90", "api_rec_k95", "api_rec_k98", "api_rec_k995", "et_24h", "wbal_k98",
              "wbal_k995", "rain24_x_sm100", "rain24_x_sm7", "api7_x_sm100", "sm255_rank", "sm100_rank"],
    "nonlin": ["log_rain24", "log_api7", "log_api30", "rain_max_h_24", "rain_max_h_72",
               "wet_hours_24", "wet_hours_168", "heavy_hours_168", "dry_spell_h"],
    "clim_idx": ["nino", "nino_lag30d", "nino_lag90d", "mjo_amp", "mjo_sin", "mjo_cos"],
    "catch": ["up_api_24h", "up_api_7d", "up_api_14d", "up_api_30d", "up_api_90d",
              "upw_api_24h", "upw_api_7d", "upw_api_14d", "upw_api_30d", "upw_api_90d",
              "up_api_rec_k98", "up_sm100", "up_sm7", "up_sm100_d30", "n_upstream"],
    "route": ["route_v50_24h", "route_v50_7d", "route_v50_14d", "route_v100_24h", "route_v100_7d",
              "route_v100_14d", "route_v200_24h", "route_v200_7d", "route_v200_14d"],
    "neigh": ["neigh_api_24h", "neigh_api_7d", "neigh_api_30d"],
    "calendar": ["month", "hour", "doy", "doy_sin1", "doy_cos1", "doy_sin2", "doy_cos2", "doy_sin3", "doy_cos3"],
    "static": ["ORD_STRA", "DIST_DN_KM", "latitude", "longitude", "built_surface_m2", "landcover_class", "log_upland"],
}
FEATS = [c for grp in FEAT_GROUPS.values() for c in grp if c in BASE.columns]

# Data latih: target bersih sampai batas, ditempeli fitur dan jangkar.
b = BASE[["datetime", "nama_pos"] + FEATS]
trn = CLEAN[CLEAN["datetime"] <= CUTOFF][["datetime", "nama_pos", "tma_mdpl"]].merge(b, on=["datetime", "nama_pos"], how="left")
trn["anchor"] = anchor_doy(CLEAN, trn)
trn = trn[trn["anchor"].notna()].reset_index(drop=True)
trn["y"] = trn["tma_mdpl"] - trn["anchor"]     # target model adalah simpangan dari jangkar

# Baris test: identitas dari test.csv, tanpa nilai jawaban.
te = pd.read_csv(os.path.join(DATA, "test.csv"))
parts = te["id"].str.split(" - ", n=1, expand=True)
ev = pd.DataFrame({"datetime": pd.to_datetime(parts[0]), "nama_pos": parts[1]})
ev = ev.merge(b, on=["datetime", "nama_pos"], how="left")
ev["anchor"] = anchor_doy(CLEAN, ev)
ev = ev.sort_values(["nama_pos", "datetime"]).reset_index(drop=True)
ev["mainriv"] = ev["nama_pos"].map(STATIC["MAIN_RIV"])
print("Baris latih:", len(trn), "| baris test:", len(ev))

Baris latih: 82750 | baris test: 21780


## 6. Model X dan skor outlier

Model X adalah pohon gradient boosting ringan yang memperkirakan simpangan tinggi air dari
jangkar. Prestasi model tidak penting di sini. Ia hanya menghasilkan nilai harapan sebagai
pembanding. Dari prediksi itu kita susun skor mencurigakan dalam tiga tahap.

Pertama, prediksi tiap pos distandarkan agar pos berfluktuasi kecil dan besar setara. Kedua,
kita ukur simpangan tiap baris terhadap rata rata pos satu sistem sungai pada waktu yang sama.
Ketiga, kita kurangi dengan rata rata simpangan tetangga waktunya agar lonjakan sesaat lebih
dihargai daripada penyimpangan yang berlangsung berhari hari. Terakhir skor dikalikan risiko
historis pos agar pos yang rawan salah catat terangkat dan pos yang tidak pernah keliru turun.

In [6]:
def train_and_score():
    # Latih Model X dengan seed terkunci pada data latih.
    Xtr = trn[FEATS + ["nama_pos"]].copy(); Xtr["nama_pos"] = Xtr["nama_pos"].astype("category")
    Xev = ev[FEATS + ["nama_pos"]].copy()
    Xev["nama_pos"] = pd.Categorical(Xev["nama_pos"], categories=Xtr["nama_pos"].cat.categories)
    model = lgb.LGBMRegressor(random_state=SEED, verbose=-1, n_jobs=-1, **LGB_PARAMS)
    model.fit(Xtr, trn["y"].values, categorical_feature=["nama_pos"])
    pred = ev["anchor"].values + model.predict(Xev)     # prediksi tinggi air

    # Standarkan prediksi per pos, lalu ukur simpangan terhadap pos satu sistem sungai.
    df = pd.DataFrame({"pos": ev["nama_pos"].values, "v": pred}); g = df.groupby("pos")["v"]
    zn = ((df["v"] - g.transform("mean")) / g.transform("std").replace(0, 1)).values
    base = np.abs(zn - ev.assign(zn=zn).groupby(["mainriv", "datetime"])["zn"].transform("mean").values)

    # Koreksi transien: kurangi rata rata simpangan tetangga waktu pos yang sama.
    k = TRANSIENT_K
    roll = ev.assign(b=base).groupby("nama_pos")["b"].transform(
        lambda x: (x.rolling(2 * k + 1, center=True, min_periods=1).sum() - x) / (2 * k)).values
    tr_sc = base - TRANSIENT_ALPHA * roll
    tr_sc = tr_sc - tr_sc.min() + 1e-6      # geser agar positif untuk perkalian

    # Kalikan dengan risiko historis pos.
    risk = ev["nama_pos"].map(RISK).fillna(0).values
    score = tr_sc * (risk + RISK_FLOOR) ** RISK_BETA

    out = ev[["nama_pos", "datetime"]].copy()
    out["pred_level"] = pred
    out["outlier_score"] = score
    out["rank"] = pd.Series(score).rank(ascending=False).astype(int).values
    return out

RESULT = train_and_score()
print("Selesai. Total baris dinilai:", len(RESULT))

Selesai. Total baris dinilai: 21780


## 7. Hasil: daftar baris paling mencurigakan

Kita tampilkan lima belas baris dengan skor tertinggi. Baris-baris inilah yang paling layak
ditinjau sebagai kemungkinan kesalahan pencatatan. Perhatikan bahwa puncak daftar cenderung
diisi lonjakan satu slot di pos berfluktuasi kecil yang historis rawan salah catat, yaitu
pola khas kesalahan input manusia, bukan peristiwa hidrologi yang sebenarnya.

In [ ]:
top = RESULT.nsmallest(15, "rank")[["rank", "nama_pos", "datetime", "pred_level", "outlier_score"]]
print("Lima belas baris paling mencurigakan menurut skor outlier:")
print(top.to_string(index=False))

Lima belas baris paling mencurigakan menurut skor outlier:
 rank                  nama_pos            datetime  pred_level  outlier_score
    1          Kali Pepe - PTPN 2025-11-03 12:00:00   83.022557       0.000610
    2          Kali Pepe - PTPN 2025-10-26 18:00:00   83.043615       0.000608
    3                    Sekayu 2026-03-01 06:00:00   88.522184       0.000603
    4              Wonogiri Dam 2026-05-18 18:00:00  134.333774       0.000602
    5          Kali Pepe - PTPN 2025-09-19 18:00:00   82.768528       0.000570
    6         Floodway Bridge C 2026-01-12 12:00:00    4.624483       0.000565
    7          Kali Pepe - PTPN 2025-10-27 06:00:00   83.053778       0.000565
    8     Kali Pepe - Tugu Boto 2026-01-15 18:00:00   94.960509       0.000561
    9          Kali Pepe - PTPN 2026-01-15 18:00:00   82.607549       0.000558
   10 Kali Anyar - Kreteg Abang 2025-11-10 12:00:00   87.125117       0.000555
   11 Kali Anyar - Kreteg Abang 2025-10-31 18:00:00   86.751302       0.

: 